# Module 297 — Reconnaissance Worksheet

This notebook follows HTB Section 3. It organizes safe information gathering for an authorized LLM application and demonstrates how to read the LLMmap ranking printed in the lesson.

It does not contact a model, install LLMmap, or run its queries. The parser uses only the sample output on the HTB page.

## Reconnaissance is mapping, not exploitation

Reconnaissance builds a picture of the application before testing attacks. Map the model, application integrations, input and output rules, multi-turn behavior, and safeguards. Record observations separately from guesses.

The model may not know application-level facts such as the real upload limit, authentication rules, or tool permissions. Verify those through the application interface or authorized documentation. Do not treat a prompt response as proof that a control exists.

## Checklist for an authorized assessment

Before sending a query, confirm the application and request limits are in scope. These categories summarize the questions HTB asks about.

In [ ]:
RECON_CHECKLIST = {
    "Model identity and capabilities": [
        "What model family does the application appear to use?",
        "Is it general-purpose or tuned for a specific domain?",
        "What tasks does the application expose?",
    ],
    "Application architecture": [
        "Does it use retrieval, tools, plugins, or function calls?",
        "Does the user interact with one model or several components?",
        "Does the application include earlier conversation turns?",
    ],
    "Input handling": [
        "Does it accept text, images, or files?",
        "What size and encoding boundaries are visible in the application?",
    ],
    "Output behavior and safeguards": [
        "How does it describe requests it will refuse?",
        "Does it stay within its purpose for benign off-topic requests?",
        "What authentication or rate limits are visible?",
        "Are input or output filters evident?",
    ],
}

for area, questions in RECON_CHECKLIST.items():
    print(f"\\n{area}")
    for question in questions:
        print(f"  - {question}")

## LLMmap and the sample ranking

The lesson's interactive workflow is: obtain LLMmap and its dependencies, run the interactive program, submit each generated query to an authorized LLM application, paste the response back, and inspect the ranked candidates.

HTB's sample ranks Meta Llama 3 8B Instruct at distance 24.8962 and Gemma 2 9B at distance 48.7247. The ordering is d_A < d_B. Read aloud: **“distance sub A is less than distance sub B.”** Candidate A has the smaller score, so the tool ranks it as the closer match.

HTB does not define the exact distance formula. The score is not a probability or percentage, and the first candidate is not proven to be the deployed model. Prompt wording, system instructions, safety filters, model version, and API wrappers can all change the response pattern.

In [ ]:
import re

# Sample output printed in the HTB lesson; this is not a live model response.
sample_output = """
[Distance: 24.8962] --> meta-llama/Meta-Llama-3-8B-Instruct
[Distance: 48.7247] google/hf_gemma-2-9b-it
[Distance: 49.7991] claude-3-5-sonnet-20240620
[Distance: 49.9209] meta-llama/Meta-Llama-3.1-8B-Instruct
[Distance: 49.9678] Qwen/Qwen2.5-0.5B-Instruct
"""

# Read each distance and model name from an LLMmap-style output line.
pattern = re.compile(
    r"\[Distance:\s*(?P<distance>\d+(?:\.\d+)?)\]\s*"
    r"(?:-->\s*)?(?P<model>[A-Za-z0-9_./-]+)"
)
ranked = [
    {"distance": float(match.group("distance")), "model": match.group("model")}
    for match in pattern.finditer(sample_output)
]

# Smaller scores appear first in the tool's ranking.
ranked.sort(key=lambda candidate: candidate["distance"])
for position, candidate in enumerate(ranked, start=1):
    print(f"{position}. {candidate['model']} — distance {candidate['distance']:.4f}")

## Record evidence and uncertainty

For each observation, record what was seen, where it came from, and how confident the conclusion is. “The assistant said it uses Model X” is an observation; it does not establish that the production deployment really uses Model X.

A distance ranking is one clue. Stronger conclusions need corroboration from an authorized source or independent behavior analysis. This worksheet stays within reconnaissance; later sections cover the attack techniques.

In [ ]:
assessment_record = {
    "authorized_application": "",
    "date_and_scope": "",
    "model_identity_claim": "",
    "identity_evidence_source": "",
    "input_types_and_limits": "",
    "conversation_history_behavior": "",
    "retrieval_or_tool_integrations": "",
    "authentication_and_rate_limits": "",
    "refusal_or_off_topic_behavior": "",
    "safeguards_observed": "",
    "uncertain_or_unverified_items": "",
}
assessment_record